In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import cv2

from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score

In [4]:
!pip install kagglehub
import kagglehub

# Download dataset
path = kagglehub.dataset_download("quangkhanh205/hockeyfight-frame")

print("Dataset path:", path)

import os

for root, dirs, files in os.walk(path):
    print(root)
    break


Using Colab cache for faster access to the 'hockeyfight-frame' dataset.
Dataset path: /kaggle/input/hockeyfight-frame
/kaggle/input/hockeyfight-frame


In [5]:
IMG_HEIGHT = 90
IMG_WIDTH = 160
SEQUENCE_LENGTH = 24

In [6]:
def load_sequences(folder, label):

    sequences = []
    labels = []

    for video in os.listdir(folder):

        video_path = os.path.join(folder, video)

        if not os.path.isdir(video_path):
            continue

        frames = sorted(os.listdir(video_path))

        clip = []

        for frame in frames:

            if not frame.endswith(".jpg"):
                continue

            frame_path = os.path.join(video_path, frame)

            img = cv2.imread(frame_path)

            if img is None:
                continue

            img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))

            img = img.astype("float32") / 255.0

            clip.append(img)

        # pad clip if fewer than SEQUENCE_LENGTH frames
        while len(clip) < SEQUENCE_LENGTH and len(clip) > 0:
            clip.append(clip[-1])

        if len(clip) >= SEQUENCE_LENGTH:
            sequences.append(clip[:SEQUENCE_LENGTH])
            labels.append(label)

    return sequences, labels

In [7]:
train_violence_path = os.path.join(path, "Train_Violence")
train_non_path = os.path.join(path, "Train_Non")

test_violence_path = os.path.join(path, "Test_Violence")
test_non_path = os.path.join(path, "Test_Non")

In [8]:
example = os.listdir(train_violence_path)[0]
print(os.listdir(os.path.join(train_violence_path, example))[:5])

['0_20.jpg', '0_5.jpg', '1_10.jpg', '1_15.jpg', '1_5.jpg']


In [9]:
train_violence_seq, train_violence_labels = load_sequences(train_violence_path, 1)
train_non_seq, train_non_labels = load_sequences(train_non_path, 0)

X_train = np.array(train_violence_seq + train_non_seq, dtype="float16")
y_train = np.array(train_violence_labels + train_non_labels)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (800, 24, 90, 160, 3)
y_train shape: (800,)


In [10]:
test_violence_seq, test_violence_labels = load_sequences(test_violence_path, 1)
test_non_seq, test_non_labels = load_sequences(test_non_path, 0)

X_test = np.array(test_violence_seq + test_non_seq, dtype="float16")
y_test = np.array(test_violence_labels + test_non_labels)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_test shape: (200, 24, 90, 160, 3)
y_test shape: (200,)


In [11]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)

Train: (640, 24, 90, 160, 3)
Validation: (160, 24, 90, 160, 3)


In [12]:
indices = np.arange(len(X_train))
np.random.shuffle(indices)

X_train = X_train[indices]
y_train = y_train[indices]

In [13]:
def build_brutnet(input_shape):

    inputs = layers.Input(shape=input_shape)

    x = layers.TimeDistributed(
        layers.Conv2D(64,(3,3),padding='same',activation='relu')
    )(inputs)

    x = layers.TimeDistributed(layers.MaxPooling2D((2,2)))(x)

    x = layers.TimeDistributed(
        layers.Conv2D(128,(3,3),padding='same',activation='relu')
    )(x)

    x = layers.TimeDistributed(layers.MaxPooling2D((2,2)))(x)

    x = layers.TimeDistributed(
        layers.Conv2D(256,(3,3),padding='same',activation='relu')
    )(x)

    x = layers.TimeDistributed(layers.MaxPooling2D((2,2)))(x)

    x = layers.TimeDistributed(
        layers.Conv2D(512,(3,3),padding='same',activation='relu')
    )(x)

    x = layers.TimeDistributed(
        layers.GlobalAveragePooling2D()
    )(x)

    x = layers.GRU(64)(x)

    x = layers.Dense(1024, activation='relu')(x)

    x = layers.Dense(1024, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    x = layers.Dense(64, activation='relu')(x)

    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs)

    return model

In [14]:
import gc
gc.collect()
tf.keras.backend.clear_session()


input_shape = (24, 90, 160, 3)

model = build_brutnet(input_shape)

optimizer = tf.keras.optimizers.Adam(learning_rate=1e-5)

model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 24, 90, 160, 3) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 24, 90, 160,    │         1,792 │
│ (TimeDistributed)               │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 24, 45, 80, 64) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 24, 45, 80,     │        73,856 │
│ (TimeDistributed)               │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 24, 22, 40,     │             0 │
│ (TimeDistributed)               │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_4              │ (None, 24, 22, 40,     │       295,168 │
│ (TimeDistributed)               │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_5              │ (None, 24, 11, 20,     │             0 │
│ (TimeDistributed)               │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_6              │ (None, 24, 11, 20,     │     1,180,160 │
│ (TimeDistributed)               │ 512)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_7              │ (None, 24, 512)        │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 64)             │       110,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1024)           │        66,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1024)           │     1,049,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,376,897 (12.88 MB)

 Trainable params: 3,376,897 (12.88 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

checkpoint = ModelCheckpoint(
    "best_brutnet_model.h5",
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=4,
    callbacks=[checkpoint, early_stop]
)

In [ ]:
def plot_loss_accuracy(history):

    plt.figure(figsize=(12,5))

    plt.subplot(1,2,1)
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title("Loss")
    plt.legend(["Train","Val"])

    plt.subplot(1,2,2)
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title("Accuracy")
    plt.legend(["Train","Val"])

    plt.show()

plot_loss_accuracy(history)

In [ ]:
best_model = tf.keras.models.load_model("best_brutnet_model.h5")

In [ ]:
y_prob = best_model.predict(X_test)

y_pred = (y_prob > 0.5).astype(int)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Non-Violence','Violence'],
    yticklabels=['Non-Violence','Violence']
)

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
print(classification_report(y_test, y_pred))

acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", acc)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

In [ ]:
print("Train shape:", X_train.shape)
print("Label distribution:", np.unique(y_train, return_counts=True))
print("Min pixel:", X_train.min())
print("Max pixel:", X_train.max())